## Human Approval Flows in Langgraph ::
- Human Approval flow means the AI does not directly perform an important action.
- -  Instead:
 -  AI prepares a response or action .
 - Human reviews it .
 - Human approv;es or rejects it .
 - Then the graph continuous .

- -  This is useful in :
 -  Sending Mails , Deploying Code , Approving the payments , updating the datbase .

In [17]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph , START , END

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph , START , END

# STATE
class ApprovalState(TypedDict):
    request: str
    generated_response: str
    human_decision: str
    final_status: str

# NODE 1: Generate AI Response

def generate_response(state: ApprovalState) -> dict: # Yeh node user ke input/request ko padhta hai aur ek AI response generate karta hai.
    # State se 'request' nikal kar ek response string bana rahe hain
    return {
        "generated_response": (
            f"AI generated response for: {state['request']}"
        )
    }



# NODE 2: Human Approval
# Yeh node terminal par AI ke response ko print karta hai aur user se input mangta hai.
# Yeh interruption ya review point ki tarah kaam karta hai.

def human_approval(state: ApprovalState) -> dict:
    print("\nGenerated Response:")
    print(state["generated_response"])

    decision = input("\nApprove this response? (yes/no): ").strip().lower()
     
     # Input validation: Jab tak user 'yes' ya 'no' na de, tab tak loop chalega
    while decision not in ["yes", "no"]:
        print("Invalid input! Please enter yes or no.")
        decision = input("Approve this response? (yes/no): ").strip().lower()

    return {
        "human_decision": decision
    }

# NODE 3: Approved
# Agar human ne 'yes' bola, toh yeh node execute hoga aur final status update karega.
def approved_node(state: ApprovalState) -> dict:
    return {
        "final_status": "Response Approved and Sent"
    }

# NODE 4: Rejected
# Agar human ne 'no' bola, toh yeh node execute hoga aur process rok dega.
def rejected_node(state: ApprovalState) -> dict:
    return {
        "final_status": "Response Rejected"
    }

# CONDITIONAL ROUTING
def route_decision(state: ApprovalState) -> Literal["approved", "rejected"]:

    if state["human_decision"] == "yes":
        return "approved"

    return "rejected"


# ==================================
# BUILD GRAPH
# ==================================

builder = StateGraph(ApprovalState)

builder.add_node("generate_response", generate_response)
builder.add_node("human_approval", human_approval)
builder.add_node("approved", approved_node)
builder.add_node("rejected", rejected_node)


# Edges
builder.add_edge(START, "generate_response") # START se pehle "generate_response" chalega

# Phir "generate_response" ke baad "human_approval" chalega
builder.add_edge("generate_response","human_approval")


# Conditional Edges
builder.add_conditional_edges( # "human_approval" ke baad route_decision function chalega jo next node decide karega
    "human_approval",
    route_decision,
    {
        "approved": "approved",
        "rejected": "rejected"
    }
)


# End Edges
builder.add_edge("approved", END)
builder.add_edge("rejected", END)


# ==================================
# COMPILE GRAPH
# ==================================

graph = builder.compile()


# ==================================
# RUN GRAPH
# ==================================

result = graph.invoke({
    "request": "Leave request for 5 days",
    "generated_response": "",
    "human_decision": "",
    "final_status": ""
})


# ==================================
# FINAL RESULT
# ==================================

print("\n========== FINAL RESULT ==========")

print("Request:", result["request"])
print("Generated Response:", result["generated_response"])
print("Human Decision:", result["human_decision"])
print("Final Status:", result["final_status"])

print("==================================")



Generated Response:
AI generated response for: Leave request for 5 days

========== FINAL RESULT ==========
Request: Leave request for 5 days
Generated Response: AI generated response for: Leave request for 5 days
Human Decision: no
Final Status: Response Rejected


==============================================================================

THEORY: ReAct (Reason + Act) Agent kya hai?
- A ReACT agent thinks about the problem,then decides what action or tool it shound use.
- instead of directly giving an answer, it works step-by-step.
==============================================================================
- ReAct pattern ka matlab hota hai **Reasoning + Action**. 
- Ismein AI model pehle query ko analyze karta hai (Sochta hai / Thought banata hai),
- phir decide karta hai ki kaun sa tool use karna chahiye (Action), aur 
- cuối mein tool ka result nikal kar final answer banata hai.

In [19]:
## reactive agent 
## State
class ReACTState(TypedDict): # Agent ke execution ke dauran jo variables track honge unka schema.
    user_query: str
    thought: str
    selected_tool: str
    tool_result: str
    final_answer: str

### Tool function
def calculator_tool(expression) -> str: # Ek custom python function jo mathematical expressions ko evaluate karta hai safely.
    try:
        return str(eval(expression)) # Python ka eval() function string math expression ko calculate karta hai (e.g., "2*4" -> 8)
    except:
        return "Invalid Calculation"

### Node-1: Reasoning
def think_node(state: ReACTState): # Yeh node user query ko dekhta hai aur decide karta hai ki kya yeh math calculation hai.
    query=state["user_query"]
    if any(char.isdigit() for char in query): # Check karte hain ki query mein koi digit (number) hai ya nahi
        return{"thought":"This looks like a math question",
                "selected_tool":"calculator"
        }
    return {"thought" : "No tool required","selected_tool" : None}

###node -2 : 
def action_node(state:ReACTState): # Yeh node check karta hai ki kaun sa tool select hua hai aur usko run karta hai.
    tool=state['selected_tool']
    query=state['user_query']
    if tool=='calculator':
        result=calculator_tool(query)
    else:
        result="General Answer"
    return{"tool_result":result}
def final_node(state:ReACTState): # Sabhi steps (Thought aur Tool Result) ko combine karke final answer banata hai.
    return{
        "final_answer":f"Thought : {state['thought']}\n Result: {state['tool_result']}"    
    }

# Nodes ko jodkar ek linear workflow banana: Start -> Think -> Action -> Final -> End
# BUILD ReACT GRAPH
builder = StateGraph(ReACTState)
builder.add_node("think_node", think_node)
builder.add_node("action_node", action_node)
builder.add_node("final_node", final_node)

builder.add_edge(START,"think_node")
builder.add_edge("think_node","action_node")
builder.add_edge("action_node","final_node")
builder.add_edge("final_node",END)

graph = builder.compile()
result=graph.invoke({
        "user_query":"2*4"})
print(result['final_answer'])

Thought : This looks like a math question
 Result: 8
